In [ ]:
# Example configuration
model_name = 'xgboost'
device = 'cpu'
min_samples_per_class = 5
n_frac = 0.01
n_trials_tpe = 10
plot_param_importances = True # can take very long!
timeout_tpe = 60
random_state = 42
n_jobs = -1

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# UTILS

In [ ]:
def stratified_sample_df(df, stratify_col, frac, min_samples_per_class, random_state):
    grouped = df.groupby(stratify_col)
    sample = grouped.apply(
        lambda x: x.sample(
            n=max(min_samples_per_class, int(len(x) * frac)), 
            replace=len(x) < max(min_samples_per_class, int(len(x) * frac)),
            random_state=random_state
        )
    ).reset_index(drop=True)
    return sample

In [ ]:
def build_classifier(n_classes, extra_kwargs={}):
    clf = None
    if model_name in ['lgbm', 'lightgbm']:
        import lightgbm as lgb
        kwargs = {
            'device_type': device,
            'n_jobs': n_jobs,
            'objective': 'binary' if n_classes == 2 else 'multiclass',
            'random_state': random_state,
            'verbose': 1,
            **extra_kwargs
        }
        if n_classes > 2:
            kwargs['num_class'] = n_classes
        clf = lgb.LGBMClassifier(**kwargs)
    elif model_name in ['xgb', 'xgboost']:
        from xgboost import XGBClassifier
        kwargs = {
            'device': device,
            'eval_metric': 'logloss' if n_classes == 2 else 'mlogloss',
            'n_jobs': n_jobs,
            'random_state': random_state,
            'tree_method': 'hist',
            **extra_kwargs
        }
        clf = XGBClassifier(**kwargs)
    elif model_name in ['xgbrf', 'xgboostrf']:
        from xgboost import XGBRFClassifier
        kwargs = {
            'device': device,
            'eval_metric': 'logloss' if n_classes == 2 else 'mlogloss',
            'n_jobs': n_jobs,
            'random_state': random_state,
            'tree_method': 'hist',
            **extra_kwargs
        }
        clf = XGBRFClassifier(**kwargs)
    return clf

In [ ]:
from collections import OrderedDict
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def process_metrics(y_test, y_pred):
    
    accuracy = accuracy_score(y_test, y_pred)
    
    precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    precision_micro = precision_score(y_test, y_pred, average='micro', zero_division=0)
    precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    
    recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    recall_micro = recall_score(y_test, y_pred, average='micro', zero_division=0)
    recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results = OrderedDict({
        'Accuracy': accuracy,
        'Precision (Macro)': precision_macro,
        'Precision (Micro)': precision_micro,
        'Precision (Weighted)': precision_weighted,
        'Recall (Macro)': recall_macro,
        'Recall (Micro)': recall_micro,
        'Recall (Weighted)': recall_weighted,
        'F1 (Macro)': f1_macro,
        'F1 (Micro)': f1_micro,
        'F1 (Weighted)': f1_weighted
    })

    pprint(results, indent=4)

In [ ]:
from sklearn.metrics import classification_report

def process_classification_report(y_test, y_pred):
    try:
        labels_str = [str(x) for x in labels]
        cr = classification_report(y_test, y_pred, digits=6, target_names=labels_str)
        print(cr)
    except Exception as e:
        print('Cloud not build/show classification report. Reason:', e)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def process_confusion_matrix(y_test, y_pred, round_thousand=False):
    try:
        y_test_decoded = [index_to_label[label] for label in y_test]
        y_pred_decoded = [index_to_label[label] for label in y_pred]
        cm = confusion_matrix(y_test_decoded, y_pred_decoded, labels=labels)
        if round_thousand:
            cm = np.round(cm / 1000, 1)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(values_format='g')
    except Exception as e:
        print('Cloud not build/show confusion matrix. Reason:', e)

In [ ]:
import numpy as np

def value_counts_to_dict(array):
    unique, counts = np.unique(array, return_counts=True)
    value_counts_dict = dict(zip(unique, counts))
    return value_counts_dict

# STEP 1: DATA PREP

In [ ]:
# %load_ext cudf.pandas
import numpy as np
import pandas as pd
from pprint import pprint

df_sample = pd.read_parquet('input/MQTT_IoT_IDS2020_PacketFeatures.parquet')

# Categorical encode Protocol column
df_sample['protocol'], protocol_values = pd.factorize(df_sample['protocol'])
protocol_to_index = {value: i for i, value in enumerate(protocol_values)}
index_to_protocol = {v: k for k, v in protocol_to_index.items()}
protocols = list(protocol_to_index.keys())
pprint(protocol_to_index, indent=4)

# Categorical encode Label column
df_sample['label'], unique_values = df_sample['label'].factorize()
label_to_index = {value: i for i, value in enumerate(unique_values)}
index_to_label = {v: k for k, v in label_to_index.items()}
labels = list(label_to_index.keys())
pprint(label_to_index, indent=4)

In [ ]:
df_sample.dtypes

In [ ]:
from dtype_diet import report_on_dataframe, optimize_dtypes

print(f'Original DF memory: {df_sample.memory_usage(deep=True).sum()/1024/1024} MB')

for col in df_sample.select_dtypes(include=['float']).columns:
    if (df_sample[col] % 1 == 0).all() and not df_sample[col].isna().any():
        print(f"Converting {col} from float to int")
        df_sample[col] = df_sample[col].astype(int)

proposed_df_sample = report_on_dataframe(df_sample, unit="MB")
df_sample = optimize_dtypes(df_sample, proposed_df_sample)

print(f'Optimized DF memory: {df_sample.memory_usage(deep=True).sum()/1024/1024} MB')

In [ ]:
df_sample.dtypes

In [ ]:
# not stratified df sampling (just to check)
df_sample['label'].sample(frac=n_frac, random_state=random_state).value_counts()

In [ ]:
# stratified df sampling
df_sample = stratified_sample_df(df_sample, 'label', n_frac, min_samples_per_class, random_state)

In [ ]:
df_sample['label'].value_counts()

In [ ]:
X_sample = df_sample.drop('label', axis=1)

print(type(X_sample))
print(X_sample.shape)
print(X_sample.dtypes)

In [ ]:
y_sample = df_sample['label']

print(type(y_sample))
print(y_sample.shape)
print(y_sample.dtype)
print(y_sample.unique())
print(y_sample.nunique())
y_sample.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

# Split into 80% training+validation and 20% test
X_sample_train_val, X_sample_test, y_sample_train_val, y_sample_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=random_state, stratify=y_sample
)

# Split the 80% training+validation set into 75% training and 25% validation
X_sample_train, X_sample_val, y_sample_train, y_sample_val = train_test_split(
    X_sample_train_val, y_sample_train_val, test_size=0.25, random_state=random_state, stratify=y_sample_train_val
)

# Verify the sizes of the splits
print(f'Training   : {len(X_sample_train)}\t({(100.0 * len(X_sample_train) / len(X_sample)):.2f}) %\t{sorted(y_sample_train.unique())}')
print(f'Validation : {len(X_sample_val)}\t({(100.0 * len(X_sample_val) / len(X_sample)):.2f}) %\t{sorted(y_sample_val.unique())}')
print(f'Test       : {len(X_sample_test)}\t({(100.0 * len(X_sample_test) / len(X_sample)):.2f}) %\t{sorted(y_sample_test.unique())}')

In [ ]:
assert sorted(y_sample_train.unique()) == sorted(y_sample_val.unique())
assert sorted(y_sample_train.unique()) == sorted(y_sample_test.unique())
assert sorted(y_sample_val.unique()) == sorted(y_sample_test.unique())

# STEP 2: PREPROCESSING & FEATURE SELECTION

### Preprocessing Methods

In [ ]:
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler, Normalizer, RobustScaler, StandardScaler

class DummyPreprocessor:
    def fit(self, _X, _y=None):
        return self
    def transform(self, _X):
        return _X

def make_preprocessor(pp_method):
    preprocessor = None
    if pp_method == 'none':
        preprocessor = DummyPreprocessor()
    if pp_method == 'maxabs':
        preprocessor = MaxAbsScaler()
    elif pp_method == 'minmax':
        preprocessor = MinMaxScaler()
    elif pp_method == 'norm':
        preprocessor = Normalizer()
    elif pp_method == 'robust':
        preprocessor = RobustScaler()
    elif pp_method == 'standard':
        preprocessor = StandardScaler()
    return preprocessor

### Feature Selection Methods

In [ ]:
from shaphypetune import BoostSearch, BoostBoruta, BoostRFE, BoostRFA

class PatchedBoostRFE(BoostRFE):
    def __init__(self, *args, **kwargs):
        self.class_idx = kwargs.pop("class_idx", 0)
        super().__init__(*args, **kwargs)

    def fit(self, X, y, eval_set=None):
        import numpy as np
        self.support_ = np.ones(X.shape[1], dtype=bool)
        current_features = np.arange(X.shape[1])

        while current_features.size > self.min_features_to_select:
            self.estimator_ = self.estimator.__class__(**self.estimator.get_params())
            self.estimator_.fit(X.iloc[:, current_features], y)
            importances = self.get_importances(X.iloc[:, current_features], y)

            ranks = np.argsort(importances)
            step = min(self.step, current_features.size)
            to_remove = current_features[ranks[:step]]
            self.support_[to_remove] = False

            current_features = np.where(self.support_)[0]

            if current_features.size <= self.min_features_to_select:
                break

        return self

    def get_importances(self, X, y):
        import shap
        if not hasattr(self, "_explainer") or self._explainer is None:
            self._explainer = shap.Explainer(self.estimator_, X)
    
        shap_values = self._explainer(X, check_additivity=False)
        values = shap_values.values
    
        if isinstance(values, list):
            values = np.abs(values[self.class_idx])
        elif values.ndim == 3:
            values = np.abs(values[:, self.class_idx, :])
        else:
            values = np.abs(values)
    
        return values.mean(axis=0)

class PatchedBoostRFA(BoostRFA):
    def __init__(self, *args, **kwargs):
        self.class_idx = kwargs.pop("class_idx", 0)
        super().__init__(*args, **kwargs)

    def get_importances(self, X, y):
        import shap
        explainer = shap.Explainer(self.estimator_, X)
        shap_values = explainer(X, check_additivity=False)

        values = shap_values.values
        if isinstance(values, list):
            values = np.abs(values[self.class_idx])
        elif values.ndim == 3:
            values = np.abs(values[:, self.class_idx, :])
        else:
            values = np.abs(values)
        return values.mean(axis=0)

class PatchedBoostBoruta(BoostBoruta):
    def __init__(self, *args, **kwargs):
        self.class_idx = kwargs.pop("class_idx", 0)
        super().__init__(*args, **kwargs)

    def get_importances(self, X, y):
        import shap
        explainer = shap.Explainer(self.estimator_, X)
        shap_values = explainer(X, check_additivity=False)

        values = shap_values.values
        if isinstance(values, list):
            values = np.abs(values[self.class_idx])
        elif values.ndim == 3:
            values = np.abs(values[:, self.class_idx, :])
        else:
            values = np.abs(values)
        return values.mean(axis=0)

In [ ]:
def make_feature_selector(n_classes, fs_method, fs_metric):
    if fs_method == 'none':
        return DummyFeatureSelector()

    model = build_classifier(n_classes)

    if fs_method == 'boruta':
        if fs_metric == 'shap_importances':
            return PatchedBoostBoruta(model, importance_type=fs_metric, perc=100, sampling_seed=random_state, class_idx=0)
        else:
            return BoostBoruta(model, importance_type=fs_metric, perc=100, sampling_seed=random_state)

    elif fs_method == 'rfa':
        if fs_metric == 'shap_importances':
            return PatchedBoostRFA(model, importance_type=fs_metric, min_features_to_select=1, sampling_seed=random_state, step=1, class_idx=0)
        else:
            return BoostRFA(model, importance_type=fs_metric, min_features_to_select=1, sampling_seed=random_state, step=1)

    elif fs_method == 'rfe':
        if fs_metric == 'shap_importances':
            return PatchedBoostRFE(model, importance_type=fs_metric, sampling_seed=random_state, min_features_to_select=1, step=1, class_idx=0)
        else:
            return BoostRFE(model, importance_type=fs_metric, sampling_seed=random_state, min_features_to_select=1, step=1)

### HPO

In [ ]:
%%time

import optuna

from optuna.samplers import GridSampler

def objective(trial):

    try:
        # preprocessing hyperparameters
        pp_method = trial.suggest_categorical('pp_method', ['none', 'maxabs', 'minmax', 'norm', 'robust', 'standard'])

        # feature selection hyperparameters
        fs_method = trial.suggest_categorical('fs_method', ['none', 'boruta', 'rfa', 'rfe'])
        fs_metric = trial.suggest_categorical('fs_metric', ['feature_importances', 'shap_importances'])
        
        # pipeline - preprocessing
        preprocessor = make_preprocessor(pp_method)
        preprocessor.fit(X_sample_train)
        X_sample_train_prep = pd.DataFrame(preprocessor.transform(X_sample_train.copy()), columns=X_sample_train.columns)
        X_sample_val_prep = pd.DataFrame(preprocessor.transform(X_sample_val.copy()), columns=X_sample_val.columns)
        X_sample_test_prep = pd.DataFrame(preprocessor.transform(X_sample_test.copy()), columns=X_sample_test.columns)
    
        # pipeline - feature selection
        n_classes = len(np.unique(y_sample_train))
        feature_selector = make_feature_selector(n_classes, fs_method, fs_metric)
        feature_selector.fit(X_sample_train_prep, y_sample_train, eval_set=[(X_sample_val_prep, y_sample_val)])
        selected_features = X_sample_train_prep.columns[feature_selector.support_]
        trial.set_user_attr('selected_features', selected_features)
        
        # pipeline - classifier fit/predict
        model = build_classifier(n_classes)
        model.fit(X_sample_train_prep[selected_features], y_sample_train)
        y_sample_pred = model.predict(X_sample_test_prep[selected_features])
    
        return f1_score(y_sample_test, y_sample_pred, average='macro', zero_division=0)
        
    except Exception as e:
        raise e
        print(e)
        raise optuna.TrialPruned()

# Perform HPO with a Grid sampler
search_space = {
    'pp_method': ['none', 'maxabs', 'minmax', 'norm', 'robust', 'standard'],
    'fs_method': ['boruta', 'rfa', 'rfe'],
    'fs_metric': ['feature_importances']#, 'shap_importances']
}
sampler = GridSampler(search_space)
n_trials_grid = np.prod([len(v) for v in search_space.values()])
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=n_trials_grid)

In [ ]:
study.trials_dataframe()

In [ ]:
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
if plot_param_importances:
    fig = optuna.visualization.plot_param_importances(study)
    fig.update_layout(width=800, height=600)
    fig.show()

In [ ]:
fig = optuna.visualization.plot_contour(study)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
best_preprocessor = make_preprocessor(study.best_trial.params['pp_method'])
best_preprocessor.fit(X_sample_train)
X_sample_train = pd.DataFrame(best_preprocessor.transform(X_sample_train.copy()), columns=X_sample_train.columns)
X_sample_val = pd.DataFrame(best_preprocessor.transform(X_sample_val.copy()), columns=X_sample_val.columns)
X_sample_test = pd.DataFrame(best_preprocessor.transform(X_sample_test.copy()), columns=X_sample_test.columns)

best_selected_features = study.best_trial.user_attrs['selected_features']
X_sample_train = X_sample_train[best_selected_features]
X_sample_val = X_sample_val[best_selected_features]
X_sample_test = X_sample_test[best_selected_features]

# STEP 3: DATA BALANCING

### Oversampling

In [ ]:
# from cuml.neighbors import KNeighborsClassifier
# from cuml.neighbors import NearestNeighbors
from imblearn.over_sampling import *

def build_oversampling_strategy(value_counts, threshold):
    n_occurences = sum([n for n in value_counts.values()])
    perfectly_balanced_occurences = int(n_occurences / len(value_counts.keys()))
    if threshold == "auto":
        n_generate = {
            class_: perfectly_balanced_occurences - occ
                    if occ < perfectly_balanced_occurences else 0
                    for class_, occ in value_counts.items()
        }
    else:
        n_generate = {
            class_: int(min(occ * threshold, perfectly_balanced_occurences - occ))
            if occ < perfectly_balanced_occurences else 0
            for class_, occ in value_counts.items()
        }
    return n_generate

def patch_oversampling_strategy(value_counts, n_generate):
    return {k : (value_counts[k] + n_generate[k]) for k in value_counts.keys()}

def make_over_sampler(over_strategy):
    over_sampler = RandomOverSampler(
        random_state=random_state, sampling_strategy=over_strategy)
    return over_sampler

### Undersampling

In [ ]:
# from cuml.neighbors import KNeighborsClassifier
# from cuml.neighbors import NearestNeighbors
from imblearn.under_sampling import *

def build_undersampling_strategy(value_counts, threshold):
    n_occurences = sum([n for n in value_counts.values()])
    perfectly_balanced_occurences = int(n_occurences / len(value_counts.keys()))
    if threshold == "auto":
        n_remove = {
            class_: occ - perfectly_balanced_occurences
                    if occ > perfectly_balanced_occurences else 0
                    for class_, occ in value_counts.items()
        }
    else:
        n_remove = {
            class_: int(min(occ * threshold, occ - perfectly_balanced_occurences))
            if occ > perfectly_balanced_occurences else 0
            for class_, occ in value_counts.items()
        }
    return n_remove

def patch_undersampling_strategy(value_counts, n_remove):
    return {k : (value_counts[k] - n_remove[k]) for k in value_counts.keys()}

def make_under_sampler(under_strategy):
    under_sampler = RandomUnderSampler(
        random_state=random_state, sampling_strategy=under_strategy)
    return under_sampler

### Combination

In [ ]:
def fit_resample(_X_train, _y_train, over_thresh, under_thresh):

    _X_names = _X_train.columns.tolist()
    _y_name = _y_train.name
    
    _X_train_copy = _X_train.copy()
    _y_train_copy = _y_train.copy()

    if over_thresh:
        value_counts = value_counts_to_dict(_y_train_copy)
        n_generate = build_oversampling_strategy(value_counts, over_thresh)
        over_strategy = patch_oversampling_strategy(value_counts, n_generate)
        over_sampler = make_over_sampler(over_strategy)
        _X_train_copy, _y_train_copy = over_sampler.fit_resample(_X_train_copy, _y_train_copy)

    if under_thresh:
        value_counts = value_counts_to_dict(_y_train_copy)
        n_remove = build_undersampling_strategy(value_counts, under_thresh)
        under_strategy = patch_undersampling_strategy(value_counts, n_remove)
        under_sampler = make_under_sampler(under_strategy)
        _X_train_copy, _y_train_copy = under_sampler.fit_resample(_X_train_copy, _y_train_copy)

    return pd.DataFrame(_X_train_copy, columns=_X_names), pd.Series(_y_train_copy, name=_y_name)

### HPO

In [ ]:
from itertools import chain

over_threshold_choices = list(chain(*[np.linspace(0, 4, num=17).round(2), ['auto']]))
under_threshold_choices = list(chain(*[np.linspace(0, 0.95, num=20).round(2), ['auto']]))

In [ ]:
%%time
def objective(trial):

    try:
        # data balancing hyperparameters
        over_threshold = trial.suggest_categorical('over_threshold', over_threshold_choices)
        under_threshold = trial.suggest_categorical('under_threshold', under_threshold_choices)
    
        # pipeline - data balancing
        X_sample_train_bal, y_sample_train_bal = fit_resample(
            X_sample_train, y_sample_train, over_threshold, under_threshold
        )
        
        # pipeline - classifier fit/predict
        n_classes = len(np.unique(y_sample_train_bal))
        model = build_classifier(n_classes)
        model.fit(X_sample_train_bal, y_sample_train_bal)
        y_sample_pred = model.predict(X_sample_test)
    
        return f1_score(y_sample_test, y_sample_pred, average='macro', zero_division=0)

    except Exception as e:
        print(e)
        raise optuna.TrialPruned()

# Perform HPO with a Grid sampler
search_space = {
    'over_threshold': over_threshold_choices,
    'under_threshold': under_threshold_choices
}
sampler = GridSampler(search_space)
n_trials_grid = np.prod([len(v) for v in search_space.values()])
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=n_trials_grid)

In [ ]:
study.trials_dataframe()

In [ ]:
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
if plot_param_importances:
    fig = optuna.visualization.plot_param_importances(study)
    fig.update_layout(width=800, height=600)
    fig.show()

In [ ]:
fig = optuna.visualization.plot_contour(study)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
best_over_threshold = study.best_trial.params['over_threshold']
best_under_threshold = study.best_trial.params['under_threshold']

X_sample_train, y_sample_train = fit_resample(
    X_sample_train, y_sample_train, best_over_threshold, best_under_threshold
)

# STEP 4: CLASSIFICATION

In [ ]:
%%time
from optuna.samplers import TPESampler

# Configure HPO objetive function
def objective(trial):

    try:
        # pipeline - classifier hyperparameters
        if model_name in ['lgbm', 'lightgbm', 'xgb', 'xgboost']:
            hpo_kwargs = {
                'n_estimators': trial.suggest_int('n_estimators', 10, 200),
                'max_depth': trial.suggest_int('max_depth', 2, 20),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
            }
        elif model_name in ['xgbrf', 'xgboostrf']:
            hpo_kwargs = {
                'n_estimators': trial.suggest_int('n_estimators', 10, 200),
                'max_depth': trial.suggest_int('max_depth', 2, 20),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 0, 1.0),
                'reg_lambda': trial.suggest_float('reg_lambda', 0, 1.0)
        }

        # pipeline - classifier fit/predict
        n_classes = len(np.unique(y_sample_train))
        model = build_classifier(n_classes, hpo_kwargs)
        model.fit(X_sample_train, y_sample_train)
        y_sample_pred = model.predict(X_sample_test)
    
        return f1_score(y_sample_test, y_sample_pred, average='macro', zero_division=0)
        
    except Exception as e:
        print(e)
        raise optuna.TrialPruned()

# Perform HPO with a TPE sampler
sampler = TPESampler()
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=n_trials_tpe, timeout=timeout_tpe)

In [ ]:
study.trials_dataframe()

In [ ]:
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
if plot_param_importances:
    fig = optuna.visualization.plot_param_importances(study)
    fig.update_layout(width=800, height=600)
    fig.show()

In [ ]:
fig = optuna.visualization.plot_contour(study)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
best_model_params = study.best_trial.params

# STEP 5: EXTRAPOLATION & GENERALIZATION

In [ ]:
df_full = pd.read_parquet('input/MQTT_IoT_IDS2020_PacketFeatures.parquet')

# Categorical encode Protocol column
df_full['protocol'].replace(to_replace=protocol_to_index, inplace=True)

# Categorical encode Label column
df_full['label'].replace(to_replace=label_to_index, inplace=True)

In [ ]:
print(f'Original DF memory: {df_full.memory_usage(deep=True).sum()/1024/1024} MB')

for col in df_full.select_dtypes(include=['float']).columns:
    if (df_full[col] % 1 == 0).all() and not df_full[col].isna().any():
        print(f"Converting {col} from float to int")
        df_full[col] = df_full[col].astype(int)

proposed_df_full = report_on_dataframe(df_full, unit="MB")
df_full = optimize_dtypes(df_full, proposed_df_full)

print(f'Optimized DF memory: {df_full.memory_usage(deep=True).sum()/1024/1024} MB')

In [ ]:
X_full = df_full.drop('label', axis=1)

print(type(X_full))
print(X_full.shape)
print(X_full.dtypes)

In [ ]:
y_full = df_full['label']

print(type(y_full))
print(y_full.shape)
print(y_full.dtype)
print(y_full.nunique())

In [ ]:
# Split into 80% training and 20% test
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=random_state, stratify=y_full
)

# Verify the sizes of the splits
print(f'Training   : {len(X_full_train)}\t({(100.0 * len(X_full_train) / len(X_full)):.2f}) %\t{sorted(y_full_train.unique())}')
print(f'Test       : {len(X_full_test)}\t({(100.0 * len(X_full_test) / len(X_full)):.2f}) %\t{sorted(y_full_test.unique())}')

In [ ]:
assert sorted(y_full_train.unique()) == sorted(y_full_test.unique())

### Baseline

In [ ]:
%%time
n_classes = len(np.unique(y_full_train))
model = build_classifier(n_classes)
model.fit(X_full_train, y_full_train)
y_full_pred = model.predict(X_full_test)

In [ ]:
process_metrics(y_full_test, y_full_pred)

In [ ]:
process_classification_report(y_full_test, y_full_pred)

In [ ]:
process_confusion_matrix(y_full_test, y_full_pred)

### Optimized

In [ ]:
import os
import tempfile

def get_model_size(model):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".ubj") as temp_model:
        model_path = temp_model.name
    if model_name in ['lgbm', 'lightgbm']:
        model.booster_.save_model(model_path)
    elif model_name in ['xgb', 'xgboost', 'xgbrf', 'xgboostrf']:
        model.save_model(model_path)
    model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
    os.remove(model_path)
    return round(model_size_mb, 2)

In [ ]:
import psutil
import time
import tracemalloc
from threading import Thread

# Function to monitor CPU usage at high frequency
def monitor_cpu(process, cpu_list, sampling_rate=1e-6):  # High-frequency sampling (1µs)
    while running:
        cpu_list.append(process.cpu_percent(interval=None))
        time.sleep(sampling_rate)  # Sleep briefly to avoid excessive overhead

# Get the current process
process = psutil.Process()
cpu_usage_fit = []
cpu_usage_pred = []

In [ ]:
# redo preprocessing
best_preprocessor.fit(X_full_train)
X_full_train = pd.DataFrame(best_preprocessor.transform(X_full_train.copy()), columns=X_full_train.columns)
X_full_test = pd.DataFrame(best_preprocessor.transform(X_full_test.copy()), columns=X_full_test.columns)

# redo feature selection
X_full_train = X_full_train[best_selected_features]
X_full_test = X_full_test[best_selected_features]

# redo data balancing
X_full_train, y_full_train = fit_resample(
    X_full_train, y_full_train, best_over_threshold, best_under_threshold
)

# redo classifier building
n_classes = len(np.unique(y_full_train))
model = build_classifier(n_classes, best_model_params)

In [ ]:
# Measure Training CPU & Memory Usage ###
running = True
cpu_thread = Thread(target=monitor_cpu, args=(process, cpu_usage_fit))
cpu_thread.start()

# Start precise memory tracking
tracemalloc.start()
mem_before_fit = process.memory_info().rss  # Total memory before (in bytes)
cpu_start_fit = process.cpu_times()  # CPU time before

start_time = time.time()
model.fit(X_full_train, y_full_train)
train_time = time.time() - start_time

cpu_end_fit = process.cpu_times()  # CPU time after
mem_after_fit = process.memory_info().rss  # Total memory after (in bytes)
train_mem, _ = tracemalloc.get_traced_memory()  # Additional memory allocated
tracemalloc.stop()

running = False
cpu_thread.join()

In [ ]:
# Calculate model size
model_size_mb = get_model_size(model)

In [ ]:
# Measure Prediction CPU & Memory Usage ###
running = True
cpu_thread = Thread(target=monitor_cpu, args=(process, cpu_usage_pred))
cpu_thread.start()

tracemalloc.start()
mem_before_pred = process.memory_info().rss
cpu_start_pred = process.cpu_times()

start_time = time.time()
y_full_pred = model.predict(X_full_test)
predict_time = time.time() - start_time

cpu_end_pred = process.cpu_times()
mem_after_pred = process.memory_info().rss
predict_mem, _ = tracemalloc.get_traced_memory()
tracemalloc.stop()

running = False
cpu_thread.join()

In [ ]:
# Compute CPU time spent
train_cpu_time = (cpu_end_fit.user + cpu_end_fit.system) - (cpu_start_fit.user + cpu_start_fit.system)
predict_cpu_time = (cpu_end_pred.user + cpu_end_pred.system) - (cpu_start_pred.user + cpu_start_pred.system)

# Compute average and peak CPU usage
train_avg_cpu = sum(cpu_usage_fit) / len(cpu_usage_fit) if cpu_usage_fit else 0
train_peak_cpu = max(cpu_usage_fit) if cpu_usage_fit else 0
predict_avg_cpu = sum(cpu_usage_pred) / len(cpu_usage_pred) if cpu_usage_pred else 0
predict_peak_cpu = max(cpu_usage_pred) if cpu_usage_pred else 0

# Compute memory usage (in MB)
train_mem_mb = train_mem / (1024 * 1024)
predict_mem_mb = predict_mem / (1024 * 1024)
train_total_mem_mb = (mem_after_fit - mem_before_fit) / (1024 * 1024)
predict_total_mem_mb = (mem_after_pred - mem_before_pred) / (1024 * 1024)

In [ ]:
# Create a DataFrame to store the results in a table format
results_df = pd.DataFrame({
    "Metric": [
        "Execution Time (sec)",
        "CPU Time (sec)",
        "Peak CPU Usage (%)",
        "Avg CPU Usage (%)",
        "Total Memory (MB)",
        "Allocated Memory (MB)"
    ],
    "Training (fit)": [
        train_time,
        train_cpu_time,
        train_peak_cpu,
        train_avg_cpu,
        train_total_mem_mb,
        train_mem_mb
    ],
    "Prediction (predict)": [
        predict_time,
        predict_cpu_time,
        predict_peak_cpu,
        predict_avg_cpu,
        predict_total_mem_mb,
        predict_mem_mb
    ]
})

# Add model size to the table
results_df.loc[len(results_df)] = ["Model Size (MB)", model_size_mb, model_size_mb]

# Print the table
results_df.round(2)

In [ ]:
process_metrics(y_full_test, y_full_pred)

In [ ]:
process_classification_report(y_full_test, y_full_pred)

In [ ]:
process_confusion_matrix(y_full_test, y_full_pred)